# Inference With Trained Model v2

Order of use:
1. set config
2. read checkpoint settings
3. examine the ROI sample
4. load model
5. run ROI-only inference
6. compare prediction against ground truth inside the ROI

This notebook is for the `Run_Training_v2.py` / `Trainer_v2.py` path.
It uses the 7-channel residual loader:
- `0:3` glossy input
- `3:6` diffuse target
- `6:7` object-selection ROI mask

To keep inference on the region of interest only, this notebook zeros the residual background outside the ROI at every DDPM step.


In [1]:
from pathlib import Path
from pprint import pprint
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm import tqdm


def find_specdiff_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd() / "Specular-Highlights" / "SpecDiff",
        Path.cwd().parent,
    ]
    for candidate in candidates:
        if (candidate / "Run_Training.py").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate SpecDiff/Run_Training.py from the current working directory.")


SPECDIFF_DIR = find_specdiff_dir()
PROJECT_ROOT = SPECDIFF_DIR.parent
if str(SPECDIFF_DIR) not in sys.path:
    sys.path.insert(0, str(SPECDIFF_DIR))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import Run_Training as rt
import Run_Training_v2 as rt2
import Diffuser as diff
from pipeline import ResidualLoader as residual_loader


def checkpoint_candidates(limit: int = 20) -> list[Path]:
    roots = [
        SPECDIFF_DIR / "models",
        SPECDIFF_DIR / "checkpoints",
        PROJECT_ROOT / "models",
        PROJECT_ROOT / "checkpoints",
    ]
    found = []
    seen = set()
    for root in roots:
        if not root.exists():
            continue
        for path in sorted(root.rglob("*.pth")):
            resolved = path.resolve()
            if resolved in seen:
                continue
            seen.add(resolved)
            found.append(resolved)
    return found[:limit]


def checkpoint_is_tensor_dict(value) -> bool:
    return isinstance(value, dict) and bool(value) and all(torch.is_tensor(v) for v in value.values())


def select_state_dict(checkpoint, prefer_ema: bool = True):
    if checkpoint_is_tensor_dict(checkpoint):
        return checkpoint, "raw_state_dict"

    if not isinstance(checkpoint, dict):
        raise TypeError(f"Unsupported checkpoint type: {type(checkpoint)!r}")

    if prefer_ema and checkpoint_is_tensor_dict(checkpoint.get("ema_model_state_dict")):
        return checkpoint["ema_model_state_dict"], "ema_model_state_dict"
    if checkpoint_is_tensor_dict(checkpoint.get("model_state_dict")):
        return checkpoint["model_state_dict"], "model_state_dict"
    if checkpoint_is_tensor_dict(checkpoint.get("state_dict")):
        return checkpoint["state_dict"], "state_dict"

    raise KeyError("Could not find a usable state dict in the checkpoint.")


def chw_to_display(chw: torch.Tensor) -> np.ndarray:
    array = chw.detach().cpu().permute(1, 2, 0).float().numpy()
    if array.min() < 0.0:
        scale = max(abs(float(array.min())), abs(float(array.max())), 1e-6)
        array = 0.5 + array / (2.0 * scale)
    return np.clip(array, 0.0, 1.0)


def mask_to_display(mask: torch.Tensor) -> np.ndarray:
    return mask.detach().cpu().squeeze().float().numpy()


def stats(name: str, tensor: torch.Tensor) -> dict[str, object]:
    return {
        "name": name,
        "shape": tuple(tensor.shape),
        "min": float(tensor.min()),
        "max": float(tensor.max()),
        "mean": float(tensor.mean()),
        "std": float(tensor.std(unbiased=False)),
        "all_finite": bool(torch.isfinite(tensor).all()),
    }


def is_cuda_runtime_error(exc: BaseException) -> bool:
    text = str(exc).lower()
    return any(token in text for token in ("cuda", "cublas", "cudnn", "out of memory", "device-side assert"))


print(f"SpecDiff dir: {SPECDIFF_DIR}")
print(f"Project root: {PROJECT_ROOT}")


/Users/27171653/Library/Python/3.12/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SpecDiff dir: /Users/27171653/Desktop/PhD/Highlight-modelling/Specular-Highlights/SpecDiff
Project root: /Users/27171653/Desktop/PhD/Highlight-modelling/Specular-Highlights


In [2]:
# Settings
CHECKPOINT_PATH = None
PREFER_EMA = True
DEVICE = "auto"  # "auto", "cpu", "cuda"
FALLBACK_TO_CPU_ON_CUDA_ERROR = True

RESIDUAL_MODEL_NAME = "can"
DATA_SPLIT = "train"
SAMPLE_INDEX = 0

THRESHOLD_METHOD = "otsu"
THRESHOLD = 0.9
SOFT_GAMMA = 1.0
KERNEL_SIZE = 3
FILL_HOLES = True

SHOW_PROGRESS = False
ZERO_BACKGROUND_EACH_STEP = True

# Manual overrides for legacy raw model.pth checkpoints.
# Fill these only when the checkpoint does not store them.
BACKBONE_MODULE = None
MODEL_NAME = None
IMAGE_SIZE = None
NOISE_STEPS = None
DEPTH = None
PARAMETERIZATION = None
TARGET_SCALE = None
RESIDUAL_MODE = None


In [3]:
def read_checkpoint_info(model_path=None, prefer_ema=True):
    checkpoint = torch.load(model_path, map_location="cpu")
    state_dict, state_key = select_state_dict(checkpoint, prefer_ema=prefer_ema)

    backbone_module = BACKBONE_MODULE if BACKBONE_MODULE is not None else checkpoint.get("backbone_module")
    model_name = MODEL_NAME if MODEL_NAME is not None else checkpoint.get("model_name")
    image_size = IMAGE_SIZE if IMAGE_SIZE is not None else checkpoint.get("image_size")
    noise_steps = NOISE_STEPS if NOISE_STEPS is not None else checkpoint.get("noise_steps")
    depth = DEPTH if DEPTH is not None else checkpoint.get("depth")
    parameterization = PARAMETERIZATION if PARAMETERIZATION is not None else checkpoint.get("parameterization", "e")
    target_scale = TARGET_SCALE if TARGET_SCALE is not None else checkpoint.get("target_scale")
    residual_mode = RESIDUAL_MODE if RESIDUAL_MODE is not None else checkpoint.get("residual_mode", "subtractive")
    residual_model_name = RESIDUAL_MODEL_NAME if RESIDUAL_MODEL_NAME is not None else checkpoint.get("residual_model_name")
    data_split = DATA_SPLIT if DATA_SPLIT is not None else checkpoint.get("split", "train")

    if backbone_module is None:
        raise ValueError("Set BACKBONE_MODULE in the config cell for this checkpoint.")
    if model_name is None:
        raise ValueError("Set MODEL_NAME in the config cell for this checkpoint.")
    if image_size is None:
        raise ValueError("Set IMAGE_SIZE in the config cell for this checkpoint.")
    if noise_steps is None:
        raise ValueError("Set NOISE_STEPS in the config cell for this checkpoint.")
    if depth is None:
        raise ValueError("Set DEPTH in the config cell for this checkpoint.")
    if target_scale is None:
        raise ValueError("Set TARGET_SCALE in the config cell for this checkpoint.")
    if residual_model_name is None:
        raise ValueError("Set RESIDUAL_MODEL_NAME in the config cell for this checkpoint.")

    parameterization = str(parameterization).lower()
    if parameterization not in {"e", "x"}:
        raise ValueError("Only 'e' and 'x' are supported in this notebook.")

    residual_mode = str(residual_mode).lower()
    if residual_mode not in {"additive", "subtractive"}:
        raise ValueError("Only 'additive' and 'subtractive' residual modes are supported.")

    target_scale = float(target_scale)
    if target_scale == 0.0:
        raise ValueError("TARGET_SCALE must be non-zero.")

    return {
        "checkpoint": checkpoint,
        "state_dict": state_dict,
        "state_key": state_key,
        "backbone_module": backbone_module,
        "model_name": model_name,
        "image_size": int(image_size),
        "noise_steps": int(noise_steps),
        "depth": int(depth),
        "parameterization": parameterization,
        "target_scale": target_scale,
        "residual_mode": residual_mode,
        "residual_model_name": str(residual_model_name),
        "data_split": str(data_split),
    }


if CHECKPOINT_PATH is None:
    print("Set CHECKPOINT_PATH in the config cell. Candidate .pth files:")
    for candidate in checkpoint_candidates():
        print(candidate)
    raise ValueError("CHECKPOINT_PATH is not set.")

checkpoint_path = Path(CHECKPOINT_PATH).expanduser().resolve()
checkpoint_info = read_checkpoint_info(checkpoint_path, prefer_ema=PREFER_EMA)

state_key = checkpoint_info["state_key"]
backbone_module = checkpoint_info["backbone_module"]
model_name = checkpoint_info["model_name"]
image_size = checkpoint_info["image_size"]
noise_steps = checkpoint_info["noise_steps"]
depth = checkpoint_info["depth"]
parameterization = checkpoint_info["parameterization"]
target_scale = checkpoint_info["target_scale"]
residual_mode = checkpoint_info["residual_mode"]
residual_model_name = checkpoint_info["residual_model_name"]
data_split = checkpoint_info["data_split"]

print(f"checkpoint: {checkpoint_path}")
print(f"state key: {state_key}")
print(f"backbone: {backbone_module}")
print(f"model: {model_name}")
print(f"image size: {image_size}")
print(f"noise steps: {noise_steps}")
print(f"depth: {depth}")
print(f"parameterization: {parameterization}")
print(f"target scale: {target_scale}")
print(f"residual mode: {residual_mode}")
print(f"residual model name: {residual_model_name}")
print(f"data split: {data_split}")


Set CHECKPOINT_PATH in the config cell. Candidate .pth files:
/Users/27171653/Desktop/PhD/Highlight-modelling/Specular-Highlights/models/LatentUNetWithAttention.pth
/Users/27171653/Desktop/PhD/Highlight-modelling/Specular-Highlights/checkpoints/new/UncertLatentUNetWithAttention_epoch_1000_TParam_e_loss_0.4252/model.pth
/Users/27171653/Desktop/PhD/Highlight-modelling/Specular-Highlights/checkpoints/new/UncertLatentUNetWithAttention_epoch_600_TParam_e_loss_0.3478/model.pth
/Users/27171653/Desktop/PhD/Highlight-modelling/Specular-Highlights/checkpoints/new/UncertLatentUNetWithAttention_epoch_800_TParam_e_loss_0.3357/model.pth
/Users/27171653/Desktop/PhD/Highlight-modelling/Specular-Highlights/checkpoints/new/UncertUNetWithAttention_epoch_1000_TParam_e_loss_0.1088/model.pth
/Users/27171653/Desktop/PhD/Highlight-modelling/Specular-Highlights/checkpoints/new/UncertUNetWithAttention_epoch_200_TParam_e_loss_0.2243/model.pth
/Users/27171653/Desktop/PhD/Highlight-modelling/Specular-Highlights/ch

ValueError: CHECKPOINT_PATH is not set.

In [4]:
loader, data_info = rt2.build_notebook_residual_loader(
    project_root=PROJECT_ROOT,
    model_name=residual_model_name,
    split=data_split,
    image_size=image_size,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    threshold_method=THRESHOLD_METHOD,
    threshold=THRESHOLD,
    soft_gamma=SOFT_GAMMA,
    kernel_size=KERNEL_SIZE,
    fill_holes=FILL_HOLES,
    residual_mode=residual_mode,
)

dataset = loader.dataset
sample = dataset[SAMPLE_INDEX]
sample_name = dataset.base_dataset.samples[SAMPLE_INDEX][2]
condition = sample[:3].unsqueeze(0)
diffuse_gt = sample[3:6].unsqueeze(0)
roi_mask = sample[6:7].unsqueeze(0)
residual_gt_full = residual_loader.compute_residual(condition, diffuse_gt, residual_mode=residual_mode)
residual_gt_roi = residual_gt_full * roi_mask

print("Loader info:")
pprint(data_info)
print(f"dataset size: {len(dataset)}")
print(f"sample index: {SAMPLE_INDEX}")
print(f"sample name: {sample_name}")
pprint(stats("condition", condition))
pprint(stats("diffuse_gt", diffuse_gt))
pprint(stats("roi_mask", roi_mask))
pprint(stats("residual_gt_full", residual_gt_full))
pprint(stats("residual_gt_roi", residual_gt_roi))


NameError: name 'residual_model_name' is not defined

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

panels = [
    ("glossy / condition", condition[0], "rgb"),
    ("ground truth diffuse", diffuse_gt[0], "rgb"),
    ("Object ROI mask", roi_mask[0, 0], "mask"),
    ("ground truth residual in ROI", residual_gt_roi[0], "rgb"),
]

for axis, (title, tensor, mode) in zip(axes.flat, panels):
    if mode == "mask":
        axis.imshow(mask_to_display(tensor), cmap="gray", vmin=0.0, vmax=1.0)
    else:
        axis.imshow(chw_to_display(tensor))
    axis.set_title(title)
    axis.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
device = rt.resolve_device(DEVICE)


def load_model(checkpoint_info, device=device):
    model = rt.build_model(
        backbone_module=checkpoint_info["backbone_module"],
        model_name=checkpoint_info["model_name"],
        image_size=checkpoint_info["image_size"],
        noise_steps=checkpoint_info["noise_steps"],
        depth=checkpoint_info["depth"],
    ).to(device)
    model.load_state_dict(checkpoint_info["state_dict"])
    model.eval()
    return model


def sample_diffusion_roi(
    model,
    input_field,
    roi_mask,
    noise_steps,
    device,
    parameterization="e",
    show_progress=False,
    zero_background_each_step=True,
):
    model.eval()
    model.to(device)
    diffuser = diff.CosSchDiffuser(noise_steps, device=device)
    roi_mask = roi_mask.to(device=device, dtype=input_field.dtype)

    with torch.no_grad():
        x_t = torch.randn_like(input_field) * roi_mask
        t_now = torch.tensor([diffuser.steps], device=device).repeat(x_t.shape[0])
        t_pre = t_now - 1
        p_bar = tqdm(range(diffuser.steps)) if show_progress else range(diffuser.steps)

        for _ in p_bar:
            model_t = (t_now - 1).clamp_min(0)

            if parameterization == "e":
                predicted_noise = model(x_t, model_t, input_field)
                if zero_background_each_step:
                    predicted_noise = predicted_noise * roi_mask
                x_t = diffuser.DDPM_sample_step(x_t, t_now, t_pre, predicted_noise)
            elif parameterization == "x":
                x0_pred = model(x_t, model_t, input_field)
                if zero_background_each_step:
                    x0_pred = x0_pred * roi_mask
                alpha_bar = diffuser.sqrt_alphas_bar[t_now]
                sigma = diffuser.sqrt_one_minus_alphas_bar[t_now].clamp_min(1e-12)
                predicted_noise = (x_t - alpha_bar * x0_pred) / sigma
                if zero_background_each_step:
                    predicted_noise = predicted_noise * roi_mask
                x_t = diffuser.DDPM_sample_step(x_t, t_now, t_pre, predicted_noise)
            else:
                raise ValueError(f"Unsupported parameterization: {parameterization}")

            if zero_background_each_step:
                x_t = x_t * roi_mask

            t_now = t_pre
            t_pre = t_pre - 1

        return x_t * roi_mask


In [ ]:
model = load_model(checkpoint_info, device=device)
print(f"model loaded on {device}")


In [ ]:
def run_inference(run_device):
    condition_run = condition.to(run_device)
    diffuse_gt_run = diffuse_gt.to(run_device)
    roi_mask_run = roi_mask.to(run_device)
    residual_gt_full_run = residual_gt_full.to(run_device)
    residual_gt_roi_run = residual_gt_roi.to(run_device)

    sampled_scaled_residual = sample_diffusion_roi(
        model=model,
        input_field=condition_run,
        roi_mask=roi_mask_run,
        noise_steps=noise_steps,
        device=run_device,
        parameterization=parameterization,
        show_progress=SHOW_PROGRESS,
        zero_background_each_step=ZERO_BACKGROUND_EACH_STEP,
    )

    predicted_residual = (sampled_scaled_residual / target_scale) * roi_mask_run
    predicted_diffuse = residual_loader.reconstruct_diffuse(
        condition_run,
        predicted_residual,
        residual_mode=residual_mode,
    )
    predicted_diffuse_clamped = predicted_diffuse.clamp(0.0, 1.0)

    roi_expand = roi_mask_run.expand_as(predicted_residual)
    residual_error_roi = (predicted_residual - residual_gt_full_run) * roi_expand
    diffuse_error_roi = (predicted_diffuse - diffuse_gt_run) * roi_expand
    diffuse_error_clamped_roi = (predicted_diffuse_clamped - diffuse_gt_run) * roi_expand

    roi_denom = roi_expand.sum().clamp_min(1.0)
    outside_mask = 1.0 - roi_expand
    outside_bg_max = float((predicted_residual.abs() * outside_mask).max())

    metrics = {
        "residual_mae_roi": float(residual_error_roi.abs().sum() / roi_denom),
        "residual_rmse_roi": float(torch.sqrt((residual_error_roi ** 2).sum() / roi_denom)),
        "diffuse_mae_roi": float(diffuse_error_roi.abs().sum() / roi_denom),
        "diffuse_mae_clamped_roi": float(diffuse_error_clamped_roi.abs().sum() / roi_denom),
        "predicted_residual_abs_max_outside_roi": outside_bg_max,
    }

    return {
        "runtime_device": str(run_device),
        "sampled_scaled_residual": sampled_scaled_residual.cpu(),
        "predicted_residual": predicted_residual.cpu(),
        "predicted_diffuse": predicted_diffuse.cpu(),
        "predicted_diffuse_clamped": predicted_diffuse_clamped.cpu(),
        "residual_gt_roi": residual_gt_roi_run.cpu(),
        "residual_error_roi": residual_error_roi.cpu(),
        "diffuse_error_roi": diffuse_error_roi.cpu(),
        "diffuse_error_clamped_roi": diffuse_error_clamped_roi.cpu(),
        "roi_mask": roi_mask_run.cpu(),
        "metrics": metrics,
    }


try:
    result = run_inference(device)
except RuntimeError as exc:
    if device.type == "cuda" and FALLBACK_TO_CPU_ON_CUDA_ERROR and is_cuda_runtime_error(exc):
        print(f"CUDA inference failed on {device}: {exc}")
        print("Retrying inference on CPU.")
        model.to("cpu")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        result = run_inference(torch.device("cpu"))
    else:
        raise

runtime_device = result["runtime_device"]
sampled_scaled_residual = result["sampled_scaled_residual"]
predicted_residual = result["predicted_residual"]
predicted_diffuse = result["predicted_diffuse"]
predicted_diffuse_clamped = result["predicted_diffuse_clamped"]
residual_gt_roi = result["residual_gt_roi"]
residual_error_roi = result["residual_error_roi"]
diffuse_error_roi = result["diffuse_error_roi"]
diffuse_error_clamped_roi = result["diffuse_error_clamped_roi"]
roi_mask_out = result["roi_mask"]
metrics = result["metrics"]

print(f"runtime device: {runtime_device}")
metrics


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 9))

panels = [
    ("glossy / condition", condition[0], "rgb"),
    ("ground truth diffuse", diffuse_gt[0], "rgb"),
    ("Object ROI mask", roi_mask_out[0, 0], "mask"),
    ("ground truth residual in ROI", residual_gt_roi[0], "rgb"),
    ("pred residual in ROI", predicted_residual[0], "rgb"),
    ("abs residual error in ROI", residual_error_roi.abs()[0], "rgb"),
    ("pred diffuse", predicted_diffuse_clamped[0], "rgb"),
    ("abs diffuse error in ROI", diffuse_error_clamped_roi.abs()[0], "rgb"),
    ("pred scaled residual in ROI", sampled_scaled_residual[0], "rgb"),
    ("sample name", roi_mask_out[0, 0], "text"),
]

for axis, (title, tensor, mode) in zip(axes.flat, panels):
    if mode == "mask":
        axis.imshow(mask_to_display(tensor), cmap="gray", vmin=0.0, vmax=1.0)
    elif mode == "text":
        axis.axis("off")
        axis.text(
            0.02,
            0.5,
            f"sample: {sample_name}\nresidual mode: {residual_mode}\nparameterization: {parameterization}",
            fontsize=12,
        )
        continue
    else:
        axis.imshow(chw_to_display(tensor))
    axis.set_title(title)
    axis.axis("off")

plt.tight_layout()
plt.show()
